# Z3-Python 03 — Tactiques et théories

**Navigation** : [Index](README.md) | [<< Z3-Python-02 Sudoku](Z3-02-Sudoku-Python.ipynb) | [Z3-Python-04 Strings >>](Z3-04-Strings-Regex-Python.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Utiliser** les tactiques Z3 pour simplifier et pretraiter les contraintes avant resolution
2. **Composer** des tactiques avec `Then`, `OrElse`, `Repeat` pour créer des pipelines de resolution
3. **Manipuler** des vecteurs de bits (`BitVec`) et comprendre la différence entre arithmetic signee et non signee
4. **Modeliser** des tableaux fonctionnels (`Array`) avec `Select` et `Store`
5. **Choisir** un solveur specialise (`SolverFor`) adapte au type de contraintes

### Prerequis
- Z3-Python 01 (Introduction) : `Solver`, `Int`, `Bool`, `Real`, `Optimize`
- Notions de representation binaire des entiers

### Duree estimee : ~40 min

***

Les notebooks précédents ont utilise le solveur Z3 de maniere directe : on ajoute des contraintes, on appelle `check()`, on obtient un modèle. En pratique, les problemes reels contiennent des centaines ou milliers de contraintes, et la **stratégie de resolution** peut faire la différence entre une reponse instantanee et un timeout. Z3 offre les **tactiques** (*tactics*) pour contrôler finement cette stratégie.

Ce notebook explore également deux théories supplementaires de Z3 : les **vecteurs de bits** (`BitVec`), essentiels pour la verification de circuits et l'analyse cryptographique, et les **tableaux** (`Array`), qui modelisent des fonctions a acces indexe.

## Source

Ce notebook s'inscrit dans la série ancrée par **Yurichev, Dennis — *SAT/SMT by Example* (édition publique du 12 février 2023, 666 pages)**, Ch. 16 Toy decompiler (Tactic, p. 463) + Ch. 5 Verification (BitVec, p. 125) — combinaison de tactiques et théories. Gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\Satisfiability and SMT Solvers\2023 - SAT_SMT_by_example.pdf` (16,45 Mo, magic bytes `%PDF`). Voir [Z3-API/README.md — Ouvrage de référence](README.md#ouvrage-de-référence) pour l'index chapitré complet de la série.

In [1]:
# Imports et verification de l'environnement
from z3 import *

print(f"Imports OK : z3-solver version {get_version_string()}")

Imports OK : z3-solver version 4.16.0


### Lecture du résultat

La sortie confirme l'environnement d'exécution : `Imports OK : z3-solver version 4.16.0`.
La version **4.16.0** de `z3-solver` est donc installée, et toutes les API utilisées dans ce
notebook (`Tactic`, `Goal`, `Then`, `BitVec`, `Array`, `SolverFor`) sont disponibles.

## 1. Tactiques Z3 : contrôler la stratégie de resolution

Une **tactique** Z3 est une fonction de transformation qui prend un ensemble de contraintes (un *goal*) et produit zero, un ou plusieurs sous-problemes (*subgoals*). Les tactiques permettent de :

- **Simplifier** les contraintes avant resolution (eliminer les redondances, normaliser les expressions)
- **Decomposer** un problème en sous-problemes independants (parallelisme)
- **Pretraiter** les contraintes pour les rendre plus digestes pour le solveur

Le flux de travail typique avec les tactiques :

1. Créer un `Goal` et y ajouter des contraintes
2. Appliquer une ou plusieurs tactiques au goal
3. Convertir le résultat en `Solver` avec `solver()` ou iterer sur les sous-but

In [2]:
# Tactique simplify : simplification syntaxique de base
x = Int('x')
y = Int('y')

g = Goal()
g.add(x + x + x == 3 * x)  # Trivialement vrai apres simplification
g.add(And(x > 0, x > 0))   # Doublon
g.add(y == x + 1)

# Appliquer la tactique simplify
result = Tactic('simplify')(g)
print("Goal avant simplification :")
for c in g:
    print(f"  {c}")
print(f"\nApres Tactic('simplify') : {len(result)} sous-goal(s)")
for i in range(len(result)):
    sg = result[i]
    print(f"  Sous-goal {i} : {sg.size()} contrainte(s)")
    for j in range(sg.size()):
        print(f"    {sg[j]}")

Goal avant simplification :
  x + x + x == 3*x
  x > 0
  x > 0
  y == x + 1

Apres Tactic('simplify') : 1 sous-goal(s)
  Sous-goal 0 : 2 contrainte(s)
    Not(x <= 0)
    y == 1 + x


### Interpretation : simplify

**Sortie obtenue** : la tactique `simplify` a elimine la tautologie `x + x + x == 3 * x` (toujours vraie), fusionne le doublon `And(x > 0, x > 0)` en une seule contrainte, et normalise les expressions.

| Tactique | Effet | Cout |
|----------|-------|------|
| `simplify` | Simplification syntaxique locale (propagation, elimination de `True`/`False`) | Très faible |
| `ctx-solver-simplify` | Simplification contextuelle (tient compte des relations entre contraintes) | Plus eleve |
| `solve-eqs` | Elimine les variables par substitution d'equations | Moyen |
| `propagate-values` | Propage les constantes | Faible |

> **Note technique** : `simplify` est rapide mais superficielle ; `ctx-solver-simplify` est plus puissante mais plus couteuse. Le choix depend de la taille du problème.

In [3]:
# Tactique ctx-solver-simplify : simplification contextuelle
# Cette tactique utilise le contexte global pour simplifier davantage.

x = Int('x')
g = Goal()
g.add(x > 10)
g.add(x > 5)     # Redondant sachant x > 10
g.add(x + 2 > 7) # Redondant sachant x > 10

# Comparaison des deux tactiques
res_simple = Tactic('simplify')(g)
res_ctx = Tactic('ctx-solver-simplify')(g)

print("Goal original :")
for c in g:
    print(f"  {c}")

print("\nApres 'simplify' :")
for i in range(len(res_simple)):
    sg = res_simple[i]
    for j in range(sg.size()):
        print(f"  {sg[j]}")

print("\nApres 'ctx-solver-simplify' :")
for i in range(len(res_ctx)):
    sg = res_ctx[i]
    for j in range(sg.size()):
        print(f"  {sg[j]}")

Goal original :
  x > 10
  x > 5
  x + 2 > 7

Apres 'simplify' :
  Not(x <= 10)
  Not(x <= 5)

Apres 'ctx-solver-simplify' :
  Not(x <= 10)
  x > 5
  x + 2 > 7


### Interpretation : simplification contextuelle

**Sortie obtenue** : sur ce goal, `simplify` produit **2 contraintes** alors que `ctx-solver-simplify` en conserve **3**. C'est donc `simplify` qui elimine ici une redondance : les contraintes `x > 5` et `x + 2 > 7` se normalisent toutes deux en `Not(x <= 5)`, et `simplify` les fusionne en une seule (deduplication **syntaxique**). Inversement, `ctx-solver-simplify` -- bien que concue pour exploiter le contexte -- ne retire ni `x > 5` ni `x + 2 > 7` (pourtant impliquees par `x > 10`) et conserve les trois contraintes.

**Points clés** :
1. `simplify` normalise chaque contrainte puis fusionne les formes **syntaxiquement identiques** (ici `x > 5` et `x + 2 > 7`) : deduplication locale, peu couteuse.
2. `ctx-solver-simplify` cible la subsomption **contextuelle** (retirer une contrainte impliquee par les autres), mais son comportement n'est pas garanti : selon la version de Z3 et la forme du goal, elle peut laisser passer des contraintes contextuellement redondantes, comme on le voit ici.
3. Les deux tactiques sont **complémentaires** plutot que hierarchisees : `simplify` attrape les doublons syntaxiques que `ctx-solver-simplify` laisse passer, et reciproquement pour les implications contextuelles. Sur de gros problemes, on les combine souvent dans un pipeline (`Then`).

> **Note technique** : le résultat de `ctx-solver-simplify` depend de la version de Z3 (tactique historique au comportement partiel). Pour une subsomption contextuelle fiable, on prefere généralement combiner `simplify` avec d'autres tactiques plutot que de s'y fier seule.

### 1.1 Composer des tactiques : `Then`, `OrElse`, `Repeat`, `TryFor`

Z3 permet de **composer** des tactiques pour créer des pipelines de resolution. Les combinateurs principaux :

| Combinateur | Effet |
|-------------|-------|
| `Then(t1, t2)` | Applique t1, puis t2 sur chaque sous-but issu de t1 |
| `OrElse(t1, t2)` | Essaie t1, si echec essaie t2 |
| `Repeat(t)` | Applique t jusqu'a point fixe |
| `TryFor(t, ms)` | Applique t avec un timeout (en millisecondes) |

In [4]:
# Composition de tactiques : pipeline simplifier -> substituer -> resoudre
x = Int('x')
y = Int('y')
z = Int('z')

g = Goal()
g.add(y == x + 5)
g.add(z == y * 2)
g.add(x > 0)
g.add(z < 30)

# Pipeline : simplifier puis eliminer les equations
pipeline = Then(Tactic('simplify'), Tactic('solve-eqs'))
result = pipeline(g)

print("Goal original :")
for c in g:
    print(f"  {c}")

print(f"\nApres Then(simplify, solve-eqs) : {len(result)} sous-goal(s)")
for i in range(len(result)):
    sg = result[i]
    print(f"  Sous-goal {i} ({sg.size()} contrainte(s)) :")
    for j in range(sg.size()):
        print(f"    {sg[j]}")

# Convertir le sous-but en expression et resoudre avec un Solver classique
expr = result[0].as_expr()
print(f"\nExpression du sous-but : {expr}")

# Pour obtenir un modele avec toutes les variables, on utilise le Solver d'origine
s = Solver()
for c in g:
    s.add(c)
print(f"Resolution complete : {s.check()}")
if s.check() == sat:
    m = s.model()
    print(f"  x = {m[x]}, y = {m[y]}, z = {m[z]}")

Goal original :
  y == x + 5
  z == y*2
  x > 0
  z < 30

Apres Then(simplify, solve-eqs) : 1 sous-goal(s)
  Sous-goal 0 (2 contrainte(s)) :
    Not(y <= 5)
    Not(y >= 15)

Expression du sous-but : And(Not(y <= 5), Not(y >= 15))
Resolution complete : sat
  x = 1, y = 6, z = 12


### Interpretation : composition de tactiques

**Sortie obtenue** : le pipeline `Then(simplify, solve-eqs)` a d'abord simplifie les contraintes, puis elimine les equations `y == x + 5` et `z == y * 2` par substitution. Les variables `x` et `z` ont ete remplacees dans les inequations restantes, ne laissant que des contraintes sur `y`. On utilise ensuite un `Solver` classique pour obtenir un modèle complet.

**Points clés** :
1. `solve-eqs` detecte les equations de la forme `var == expr` et substitue `var` partout.
2. La composition `Then` execute les tactiques **en sequence** : la sortie de l'une alimente l'autre.
3. `as_expr()` convertit un sous-but en expression Z3 (conjonction de ses contraintes).
4. Les tactiques transforment les contraintes mais ne resolvent pas ; la resolution finale se fait via un `Solver`.

In [5]:
# OrElse et Repeat : essayer plusieurs strategies et iterer
x = Int('x')

# Repeat : appliquer une tactique jusqu'a point fixe
g = Goal()
g.add(x + 0 == x)
g.add(x * 1 == x)

repeat_tac = Repeat(Then(Tactic('simplify'), Tactic('propagate-values')))
result = repeat_tac(g)
print("Apres Repeat(simplify + propagate-values) :")
for i in range(len(result)):
    sg = result[i]
    print(f"  Sous-goal {i} : {sg.size()} contrainte(s)")
    for j in range(sg.size()):
        print(f"    {sg[j]}")

# OrElse : fallback entre tactiques
g2 = Goal()
g2.add(x > 5)
g2.add(x < 10)

# Essaye ctx-solver-simplify, sinon simplify en fallback
fallback = OrElse(Tactic('ctx-solver-simplify'), Tactic('simplify'))
result2 = fallback(g2)
print("\nApres OrElse(ctx-solver-simplify, simplify) :")
for i in range(len(result2)):
    sg = result2[i]
    for j in range(sg.size()):
        print(f"  {sg[j]}")

print(f"\n{len(result2)} sous-goal(s) traite(s) avec succes")

Apres Repeat(simplify + propagate-values) :
  Sous-goal 0 : 0 contrainte(s)

Apres OrElse(ctx-solver-simplify, simplify) :
  Not(x <= 5)
  x < 10

1 sous-goal(s) traite(s) avec succes


### Interpretation : combinateurs

**Sortie obtenue** : les expressions triviales `x + 0 == x` et `x * 1 == x` sont reduites par le pipeline repete. Le combinateur `OrElse` tente la première tactique et bascule sur la seconde en cas d'echec.

| Combinateur | Analogie | Usage typique |
|-------------|----------|---------------|
| `Then` | Pipeline (pipe `\|`) | Pretraitement en étapes successives |
| `OrElse` | Try-catch | Tactique rapide d'abord, robuste en fallback |
| `Repeat` | Boucle while | Raffiner iterativement jusqu'a stabilite |
| `TryFor` | Timeout | Limiter le temps passe dans une tactique couteuse |

## 2. Théorie des vecteurs de bits : `BitVec`

Un **vecteur de bits** (*bitvector*) est une sequence de n bits representant un entier modulo $2^n$. Contrairement a `Int` (entiers mathematiques non bornes), les `BitVec` modelisent le comportement exact du materiel informatique : debordement, arithmetic modulaire, opérations bit a bit.

Applications : verification de circuits, analyse cryptographique, emulation de processeurs.

| Aspect | `Int` | `BitVec(n)` |
|--------|-------|-------------|
| Domaine | $\mathbb{Z}$ (entiers infinis) | $\{0, \ldots, 2^n - 1\}$ (modulo $2^n$) |
| Debordement | Pas de debordement | Wrapping modulaire |
| Opérations bit a bit | Non disponibles | `&`, `\|`, `^`, `~`, `<<`, `>>` |

In [6]:
# BitVec : creation, arithmetic modulaire et operations bit a bit
x = BitVec('x', 8)  # Vecteur de 8 bits (0 a 255)
y = BitVec('y', 8)

s = Solver()

# Arithmetic modulaire : 255 + 1 = 0 en 8 bits (wrapping)
s.add(x == BitVecVal(255, 8))
s.add(y == x + 1)  # 255 + 1 = 0 (mod 256)

print(f"Resolution : {s.check()}")
m = s.model()
print(f"  x = {m[x]} (valeur entiere : {m[x].as_long()})")
print(f"  y = x + 1 = {m[y]} (valeur entiere : {m[y].as_long()})")
print(f"  255 + 1 modulo 256 = {(255 + 1) % 256}")

# Operations bit a bit
s2 = Solver()
a = BitVec('a', 8)
b = BitVec('b', 8)

# Trouver a et b tels que a XOR b = 0xFF mais a AND b = 0
s2.add(a ^ b == BitVecVal(0xFF, 8))  # Tous les bits differents
s2.add(a & b == BitVecVal(0, 8))      # Aucun bit en commun

print(f"\nCas XOR/AND : {s2.check()}")
m2 = s2.model()
print(f"  a = {m2[a].as_long():3d} (binaire : {m2[a].as_long():08b})")
print(f"  b = {m2[b].as_long():3d} (binaire : {m2[b].as_long():08b})")
print(f"  a XOR b = {m2[a].as_long() ^ m2[b].as_long():08b}")
print(f"  a AND b = {m2[a].as_long() & m2[b].as_long():08b}")

Resolution : sat
  x = 255 (valeur entiere : 255)
  y = x + 1 = 0 (valeur entiere : 0)
  255 + 1 modulo 256 = 0

Cas XOR/AND : sat
  a = 254 (binaire : 11111110)
  b =   1 (binaire : 00000001)
  a XOR b = 11111111
  a AND b = 00000000


### Interprétation : vecteurs de bits

**Sortie obtenue** : `x = 255 (valeur entiere : 255)` puis `y = x + 1 = 0 (valeur entiere : 0)`, avec la ligne de contrôle `255 + 1 modulo 256 = 0`. Ce zéro n'est pas un échec du solveur : un `BitVec` de 8 bits ne peut pas représenter 256, donc l'addition revient à 0 et le modèle satisfait les deux contraintes. Le second bloc imprime `a = 254 (binaire : 11111110)`, `b =   1 (binaire : 00000001)`, `a XOR b = 11111111` et `a AND b = 00000000` : les deux opérateurs s'appliquent bit par bit, et comme le bit de poids faible de `a` vaut 0, le `XOR` avec 1 le fait passer à 1 alors que le `AND` avec 1 le laisse à 0 — d'où 255 et 0 en décimal.

**Points clés** :
1. Les opérations arithmétiques sur `BitVec` sont **automatiquement modulaires** : pas de débordement, le résultat "wraps" autour de $2^n$.
2. Les opérateurs Python `&`, `|`, `^`, `~`, `<<`, `>>` sont surchargés pour construire des expressions symboliques Z3.
3. `BitVecVal(255, 8)` crée une constante ; `BitVec('x', 8)` crée une variable symbolique.


In [7]:
# Arithmetic signee vs non signee sur les BitVec
# Un meme vecteur de bits peut etre interprete comme signe ou non signe.

a = BitVec('a', 8)
b = BitVec('b', 8)

# Comparaisons signees : traite le bit de poids fort comme signe
# -128 a 127 en signe, 0 a 255 en non signe
s = Solver()
s.add(a == BitVecVal(200, 8))  # 200 en non signe = -56 en signe

# En non signe : 200 > 100 est vrai
s.add(UGT(a, BitVecVal(100, 8)))  # UGT = Unsigned Greater Than

# En signe : -56 > 100 est faux
print(f"200 (unsigned) > 100 ? {s.check()}")
if s.check() == sat:
    m = s.model()
    print(f"  a = {m[a].as_long()} (unsigned) = {m[a].as_signed_long()} (signed)")

# Tableau des operateurs de comparaison
print("\nOperateurs signes vs non signes :")
print("| Operation     | Signe     | Non signe  |")
print("|---------------|-----------|------------|")
print("| a < b         | a < b     | ULT(a, b)  |")
print("| a <= b        | a <= b    | ULE(a, b)  |")
print("| a > b         | a > b     | UGT(a, b)  |")
print("| a >= b        | a >= b    | UGE(a, b)  |")
print("| a / b         | a / b     | UDiv(a, b) |")
print("| a % b         | a % b     | URem(a, b) |")
print("| a >> b        | arith.    | LShR(a, b) |")

200 (unsigned) > 100 ? sat
  a = 200 (unsigned) = -56 (signed)

Operateurs signes vs non signes :
| Operation     | Signe     | Non signe  |
|---------------|-----------|------------|
| a < b         | a < b     | ULT(a, b)  |
| a <= b        | a <= b    | ULE(a, b)  |
| a > b         | a > b     | UGT(a, b)  |
| a >= b        | a >= b    | UGE(a, b)  |
| a / b         | a / b     | UDiv(a, b) |
| a % b         | a % b     | URem(a, b) |
| a >> b        | arith.    | LShR(a, b) |


### Interprétation : signe vs non signé

**Sortie obtenue** : la requête `200 (unsigned) > 100 ? sat` répond `sat`, et le modèle imprime `a = 200 (unsigned) = -56 (signed)`. Les deux écritures désignent le **même** mot de 8 bits, `11001000` : lu comme entier non signé il vaut 200, lu en complément à deux il vaut $200 - 256 = -56$. C'est exactement pourquoi la même valeur répond « supérieure à 100 » ou « négative » selon l'interprétation retenue — et pourquoi l'opérateur de comparaison doit être choisi explicitement. Le tableau imprimé par la cellule donne la correspondance complète : `a < b` (signé) face à `ULT(a, b)` (non signé), `a / b` face à `UDiv(a, b)`, `a >> b` arithmétique face à `LShR(a, b)`.

**Points clés** :
1. Les opérateurs Python `<`, `>`, `<=`, `>=` sont **signés** par défaut dans Z3.
2. Pour les comparaisons non signées, utiliser `UGT`, `ULT`, `UGE`, `ULE`.
3. La division et le décalage existent aussi en versions signées et non signées.


In [8]:
# Casse-tete BitVec : trouver un nombre dont les bits sont un palindrome
# Un palindrome binaire 8 bits se lit de la meme facon de gauche a droite et de droite a gauche.
# Exemple : 10011001 (= 153) est un palindrome binaire.

def reverse_bits_8(val):
    """Inverse l'ordre des 8 bits d'un BitVec 8 bits."""
    result = BitVecVal(0, 8)
    for i in range(8):
        # Extraire le bit i et le placer en position (7 - i)
        bit_i = Extract(i, i, val)
        result = result | Concat(BitVecVal(0, 7), bit_i) << (7 - i)
    return result

x = BitVec('x', 8)
s = Solver()

# x doit etre un palindrome binaire : x == reverse(x)
s.add(x == reverse_bits_8(x))
# Exclure 0 et 255 pour trouver un cas non trivial
s.add(x != BitVecVal(0, 8))
s.add(x != BitVecVal(255, 8))
# Preferer un palindrome pair (bit 0 = 0)
s.add((x & BitVecVal(1, 8)) == BitVecVal(0, 8))

print(f"Palindrome binaire : {s.check()}")
if s.check() == sat:
    m = s.model()
    val = m[x].as_long()
    print(f"  x = {val} (binaire : {val:08b})")
    print(f"  Verification : {val:08b} se lit identiquement dans les deux sens")

Palindrome binaire : sat
  x = 66 (binaire : 01000010)
  Verification : 01000010 se lit identiquement dans les deux sens


### Lecture du résultat : le palindrome binaire rendu

La sortie donne `x = 66 (binaire : 01000010)` avec le verdict `sat`. Verification par recomposition : 01000010 en base 2 vaut 64 + 2 = 66, coherente avec la valeur decimale imprimee. La symetrie se lit position par position : bit 7 = bit 0 = 0, bit 6 = bit 1 = 1, bit 5 = bit 2 = 0, bit 4 = bit 3 = 0 — le mot se retourne sur lui-même.

Point important sur ce que ce `sat` signifie : les contraintes n'exigeaient pas le plus petit palindrome pair non trivial, seulement un palindrome pair différent de 0 et de 255. D'autres modèles satisfont le même système — par exemple 00100100 (soit 36), pair et symetrique lui aussi. Z3 produit un modèle satisfaisant arbitraire, pas un minimum ; obtenir le plus petit exigerait une optimisation (`Optimize` avec `minimize`), absente de cette cellule.

Non mesure : le nombre total de modèles satisfaisants (la sortie n'en expose qu'un exemplaire).

## 3. Théorie des tableaux : `Array`

En Z3, un **tableau** (*Array*) est une fonction totale des index vers des valeurs. Les deux opérations fondamentales :

- **`Select(a, i)`** : lecture de la valeur a l'index `i` dans le tableau `a` (équivalent de `a[i]`)
- **`Store(a, i, v)`** : ecriture de la valeur `v` a l'index `i`, renvoyant un **nouveau** tableau (immutabilite)

La théorie des tableaux est essentielle pour modeliser des memoires, des caches, des registres ou des mappings.

| Propriete | Description |
|-----------|-------------|
| Immutabilite | `Store` ne modifie pas le tableau ; il en renvoie un nouveau |
| Totalite | Tout index a une valeur (la valeur par defaut si non initialise) |
| Axiome de sélection | `Select(Store(a, i, v), i) == v` |
| Axiome de non-sélection | `i != j => Select(Store(a, i, v), j) == Select(a, j)` |

In [9]:
# Array : lecture, ecriture et raisonnement sur les tableaux
# Modelisons un tableau d'entiers indexe par des entiers.

a = Array('a', IntSort(), IntSort())  # Tableau : Int -> Int

s = Solver()

# Contraintes sur le tableau initial
s.add(Select(a, 0) == 10)
s.add(Select(a, 1) == 20)
s.add(Select(a, 2) == 30)

# Demonstration : a[0] + a[1] == 30
somme = Select(a, 0) + Select(a, 1)
s.add(somme == 30)

print(f"Resolution : {s.check()}")
if s.check() == sat:
    m = s.model()
    print(f"  a[0] = {m.evaluate(Select(a, 0))}")
    print(f"  a[1] = {m.evaluate(Select(a, 1))}")
    print(f"  a[2] = {m.evaluate(Select(a, 2))}")
    print(f"  a[0] + a[1] = {m.evaluate(somme)}")

# Store : creer un nouveau tableau avec une modification
b = Store(a, 0, 99)  # b est un tableau ou l'index 0 vaut 99
s2 = Solver()
s2.add(Select(a, 0) == 10)
s2.add(Select(b, 0) == 99)       # b[0] = 99 (modifie)
s2.add(Select(b, 1) == 20)       # b[1] = 20 (inchange)

print(f"\nStore : {s2.check()}")
if s2.check() == sat:
    m2 = s2.model()
    print(f"  b[0] = {m2.evaluate(Select(b, 0))} (modifie par Store)")
    print(f"  b[1] = {m2.evaluate(Select(b, 1))} (inchange par Store)")

Resolution : sat
  a[0] = 10
  a[1] = 20
  a[2] = 30
  a[0] + a[1] = 30

Store : sat
  b[0] = 99 (modifie par Store)
  b[1] = 20 (inchange par Store)


### Interprétation : tableaux Z3

**Sortie obtenue** : le modèle fixe `a[0] = 10`, `a[1] = 20`, `a[2] = 30`, et la contrainte imprimée `a[0] + a[1] = 30` la vérifie arithmétiquement ($10 + 20 = 30$). Le bloc `Store` imprime ensuite `b[0] = 99 (modifie par Store)` et `b[1] = 20 (inchange par Store)` : l'écriture ne touche qu'un seul index, la valeur des autres est **conservée** — c'est l'axiome lecture-après-écriture que Z3 exploite pour raisonner sur mémoire et registres, sans copier le tableau.

**Points clés** :
1. Les tableaux Z3 sont des **fonctions pures** : `Store` ne mute pas, il produit une nouvelle fonction.
2. `Select(a, i)` équivaut à l'application fonctionnelle `a(i)`.
3. L'axiome fondamental : `Select(Store(a, i, v), j)` vaut `v` si `i == j`, sinon `Select(a, j)`.


In [10]:
# Array avance : contraintes sur un tableau avec Store successifs
# Scenario : un tableau represente les soldes de comptes.
# On effectue des operations et on veut verifier des proprietes.

comptes = Array('comptes', IntSort(), IntSort())

s = Solver()

# Etat initial : comptes[0] = 100, comptes[1] = 200, comptes[2] = 50
s.add(Select(comptes, 0) == 100)
s.add(Select(comptes, 1) == 200)
s.add(Select(comptes, 2) == 50)

# Operation 1 : transfer de 30 du compte 1 vers le compte 0
apres_transfer = Store(
    Store(comptes, 1, Select(comptes, 1) - 30),
    0, Select(comptes, 0) + 30
)

# Verifier que le total est conserve
total_avant = Select(comptes, 0) + Select(comptes, 1) + Select(comptes, 2)
total_apres = Select(apres_transfer, 0) + Select(apres_transfer, 1) + Select(apres_transfer, 2)

s.add(total_avant == total_apres)

print(f"Conservation du total : {s.check()}")
if s.check() == sat:
    m = s.model()
    print(f"  Avant  : c0={m.evaluate(Select(comptes, 0))}, "
          f"c1={m.evaluate(Select(comptes, 1))}, "
          f"c2={m.evaluate(Select(comptes, 2))}")
    print(f"  Apres  : c0={m.evaluate(Select(apres_transfer, 0))}, "
          f"c1={m.evaluate(Select(apres_transfer, 1))}, "
          f"c2={m.evaluate(Select(apres_transfer, 2))}")
    print(f"  Total avant = {m.evaluate(total_avant)}")
    print(f"  Total apres = {m.evaluate(total_apres)}")

Conservation du total : sat
  Avant  : c0=100, c1=200, c2=50
  Apres  : c0=130, c1=170, c2=50
  Total avant = 350
  Total apres = 350


### Lecture du résultat : la conservation du total après transfert

La sortie imprime les deux etats et les deux totaux : `Avant  : c0=100, c1=200, c2=50`, `Apres  : c0=130, c1=170, c2=50`, puis `Total avant = 350` et `Total apres = 350`. Recomposition des deltas : c0 gagne +30 (100 vers 130), c1 perd -30 (200 vers 170), c2 reste a 50. Les deux variations s'annulent (+30 - 30 = 0), ce que les totaux confirment : 100 + 200 + 50 = 350 et 130 + 170 + 50 = 350.

Ce que ce `sat` prouve exactement : les contraintes fixaient l'etat initial (comptes[0] = 100, comptes[1] = 200, comptes[2] = 50) puis exigeaient l'egalite des deux totaux. La reponse atteste la coherence de cette egalite sur l'etat concret impose — les valeurs fixees determinent déjà les deux totaux. Ce n'est pas une preuve pour tout etat initial possible : pour cela, il faudrait refuter la negation de la propriete sur des variables libres, le solveur devant alors repondre `unsat`.

Non mesure : aucun temps de resolution ni taille du problème ne sont imprimes par cette cellule.

## 4. Solveurs specialises : `SolverFor`

Z3 dispose de solveurs optimises pour des **fragments** spécifiques de la logique. Plutot que d'utiliser le solveur generique `Solver()`, on peut sélectionner un solveur adapte a la théorie concernee :

| Logique | Description | Types principaux |
|---------|-------------|------------------|
| `QF_LIA` | Quantifier-Free Linear Integer Arithmetic | `Int` (lineaire) |
| `QF_LRA` | Quantifier-Free Linear Real Arithmetic | `Real` (lineaire) |
| `QF_BV` | Quantifier-Free BitVectors | `BitVec` |
| `QF_UFLIA` | QF_LIA + fonctions non interpretees | `Int` + `Array` |

Le prefixe **QF** signifie *Quantifier-Free* (sans quantificateurs `ForAll`/`Exists`). Les solveurs specialises sont souvent **plus rapides** car ils exploitent la structure du fragment.

In [11]:
# Comparaison : Solver generique vs SolverFor specialise
import time

# Generer un systeme de contraintes lineaires entieres
n_vars = 50
variables = [Int(f'x{i}') for i in range(n_vars)]

constraints = []
# Chaque variable entre 0 et 100
for v in variables:
    constraints.append(v >= 0)
    constraints.append(v <= 100)
# Contraintes lineaires entre variables consecutives
for i in range(n_vars - 1):
    constraints.append(variables[i] + variables[i + 1] <= 150)
    constraints.append(variables[i] + variables[i + 1] >= 50)

# Solveur generique
s_generic = Solver()
for c in constraints:
    s_generic.add(c)

t0 = time.perf_counter()
result_generic = s_generic.check()
time_generic = time.perf_counter() - t0

# Solveur specialise QF_LIA
s_specialized = SolverFor('QF_LIA')
for c in constraints:
    s_specialized.add(c)

t0 = time.perf_counter()
result_specialized = s_specialized.check()
time_specialized = time.perf_counter() - t0

print(f"Solveur generique : {result_generic} en {time_generic*1000:.1f} ms")
print(f"SolverFor QF_LIA  : {result_specialized} en {time_specialized*1000:.1f} ms")
print(f"Rapport (generic / specialized) : {time_generic / max(time_specialized, 1e-9):.1f}x")

# Configuration globale avec set_option
print("\nConfiguration globale avec set_option :")
set_option('verbose', 0)       # Niveau de verbosite (0 = silencieux)
set_option('timeout', 10000)   # Timeout en millisecondes
print("  verbose=0, timeout=10000 ms configures")

Solveur generique : sat en 6.0 ms
SolverFor QF_LIA  : sat en 2.6 ms
Rapport (generic / specialized) : 2.3x

Configuration globale avec set_option :
  verbose=0, timeout=10000 ms configures


### Interpretation : solveurs specialises

**Sortie obtenue** : les deux solveurs trouvent `sat`, mais le solveur specialise `QF_LIA` est généralement plus rapide sur les contraintes lineaires entieres sans quantificateurs.

**Points clés** :
1. `SolverFor('QF_LIA')` selectionne le moteur optimise pour l'arithmetic lineaire entiere sans quantificateurs.
2. `SolverFor('QF_BV')` est adapte aux problemes de vecteurs de bits.
3. `set_option('timeout', ms)` permet de limiter le temps de calcul global.
4. Le gain de performance depend du problème ; sur de petites instances, la différence est souvent negligeable.

## Exemples guidés et exercices

Trois exercices pour vous approprier le matériel. Les stubs s'exécutent tels quels (ils rendent `None`) : complétez-les, puis relancez la cellule.

Les trois premiers énoncés ont été résolus par un rendu de TP (PR étudiante #25642) : chaque résolution est conservée comme **exemple guidé crédité**, suivie d'un exercice neuf sur le même sujet.

## Exemple guidé 1 — Composer un pipeline de tactiques (crédité #25642 — groupe 3 MIS : Alizée, Clémence et Pauline)

> Résolution originale : **groupe 3** (TP2, MIS, EPF), PR étudiante [#25642](https://github.com/jsboige/CoursIA/pull/25642). Code repris tel quel.

### Enonce

On dispose du système de contraintes suivant :
- `a == b + 1`
- `c == a * 2`
- `b > 5`
- `c < 20`

Créez un pipeline de tactiques `Then(simplify, solve-eqs)` et appliquez-le a un `Goal` contenant ces contraintes. Affichez les sous-but obtenus, puis resolvez le système avec un `Solver` pour obtenir les valeurs de `a`, `b` et `c`.

> **Indices :**
- Créez un `Goal()`, ajoutez les 4 contraintes.
- Construisez `Then(Tactic('simplify'), Tactic('solve-eqs'))` et appliquez-le au goal.
- Utilisez `len(result)` pour le nombre de sous-but et `result[i].size()` pour le nombre de contraintes.
- Pour obtenir les valeurs, ajoutez les contraintes originales a un `Solver` et appelez `check()` puis `model()`.

In [12]:
# EXEMPLE GUIDE 1 : Pipeline de tactiques (resolution groupe 3 MIS, TP2, #25642).

from z3 import *

def pipeline_tactiques() -> dict:
    # Etape 1 : variables
    a, b, c = Ints('a b c')

    # Etape 2 : goal avec les 4 contraintes de l'énoncé
    contraintes = [a == b + 1, c == a * 2, b > 5, c < 20]
    goal = Goal()
    goal.add(*contraintes)

    # Etape 3 : pipeline simplify puis solve-eqs
    tactic = Then(Tactic('simplify'), Tactic('solve-eqs'))
    result = tactic(goal)

    # Etape 4 : affichage des sous-buts
    print("Nombre de sous-buts :", len(result))
    for i in range(len(result)):
        print(f"Sous-but {i} ({result[i].size()} contraintes) :", result[i])

    nb_contraintes_final = result[0].size()

    # Etape 5 : résolution avec les contraintes ORIGINALES
    solver = Solver()
    solver.add(*contraintes)

    if solver.check() == sat:
        m = solver.model()
        a_val, b_val, c_val = m[a].as_long(), m[b].as_long(), m[c].as_long()
    else:
        a_val = b_val = c_val = None

    return {'a': a_val, 'b': b_val, 'c': c_val,
            'nb_contraintes_final': nb_contraintes_final}

resultat = pipeline_tactiques()
print("Resultat du pipeline :", resultat)

Nombre de sous-buts : 1
Sous-but 0 (2 contraintes) : [Not(a <= 6), Not(a >= 10)]
Resultat du pipeline : {'a': 7, 'b': 6, 'c': 14, 'nb_contraintes_final': 2}


### Lecture du résultat

Le pipeline rend **1 sous-but** de **2 contraintes**, `[Not(a <= 6), Not(a >= 10)]` : `solve-eqs` a substitué `b` et `c` (leurs équations disparaissent du goal), et les inégalités `b > 5` et `c < 20` se réexpriment en bornes sur `a` seul. La résolution avec un `Solver` sur les contraintes **originales** donne `{'a': 7, 'b': 6, 'c': 14, 'nb_contraintes_final': 2}` : on vérifie au passage `7 = 6 + 1`, `14 = 7 * 2`, `6 > 5` et `14 < 20`. Le sous-but simplifié sert à diagnostiquer le système ; les valeurs viennent du solveur complet.

## Exercice 1 : Ce que solve-eqs élimine vraiment

### Enonce

On dispose du système de contraintes suivant :
- `x == 2`
- `y == x + 3`
- `z > y`
- `z < 12`
- `w == x * y`

Écrivez `analyser_pipeline()` qui applique le même pipeline `Then(simplify, solve-eqs)` a un `Goal` contenant ces contraintes, et renvoie `{'nb_sous_buts': int, 'nb_contraintes': int, 'variables_restantes': list}` — la liste étant **triée** et contenant les noms des variables qui apparaissent encore dans le(s) sous-but(s). Quelles variables le pipeline a-t-il fait disparaître, et pourquoi celles-là ?

> **Indices :**
- `result[0][p]` donne la contrainte `p` du premier sous-but ; `result[0].size()` en donne le nombre.
- `z3util.get_vars(expr)` renvoie la liste des variables d'une expression Z3 ; il traverse le `Not(...)` sans souci.
- Une équation `v == terme` ne peut être éliminée que si `v` n'apparaît pas dans `terme`.

In [13]:
# EXERCICE 1 : Ce que solve-eqs elimine vraiment.

def analyser_pipeline() -> dict:
    """Applique Then(simplify, solve-eqs) et analyse ce qui subsiste dans le sous-but.

    Retourne : {'nb_sous_buts': int, 'nb_contraintes': int, 'variables_restantes': list triee}

    # Indice : creez un Goal avec les 5 contraintes, appliquez le pipeline, inspectez le sous-but.
    # Etape 1 : declarer x, y, z, w comme Int
    # Etape 2 : creer le Goal et ajouter les 5 contraintes de l'enonce
    # Etape 3 : appliquer Then(Tactic('simplify'), Tactic('solve-eqs')) et compter sous-buts et contraintes
    # Etape 4 : collecter les variables restantes avec z3util.get_vars sur chaque contrainte du sous-but
    """
    # TODO etudiant : implémentez l'analyse du pipeline
    return None  # TODO etudiant : remplacer par le résultat

resultat = analyser_pipeline()
print("Analyse du pipeline :", resultat)

# Tests attendus (decommentez apres implementation) :
# assert resultat['nb_sous_buts'] == 1
# assert resultat['nb_contraintes'] == 2
# assert resultat['variables_restantes'] == ['z']

Analyse du pipeline : None


## Exemple guidé 2 — Clef cryptographique avec BitVec (crédité #25642 — groupe 3 MIS : Alizée, Clémence et Pauline)

> Résolution originale : **groupe 3** (TP2, MIS, EPF), PR étudiante [#25642](https://github.com/jsboige/CoursIA/pull/25642). Code repris tel quel.

### Enonce

Un message de 8 bits a ete chiffre par XOR avec une clef `k`. Le message chiffre est `0xA5` (165 en decimal). On sait que le message original est un multiple de 4 (en valeur entiere non signee).

Trouvez la clef `k` (8 bits) telle que `message ^ k == 0xA5` et `message` soit un multiple de 4.

> **Indices :**
- Un multiple de 4 en binaire se termine par `00` (les 2 bits de poids faible sont nuls).
- Contrainte : `(message & BitVecVal(3, 8)) == BitVecVal(0, 8)`.
- Le chiffrement XOR est reversible : `message = 0xA5 ^ k`.

In [14]:
# EXEMPLE GUIDE 2 : Trouver la clef XOR (resolution groupe 3 MIS, TP2, #25642).

def trouver_clef_xor() -> dict:
    # Etape 1 : clef sur 8 bits
    k = BitVec('k', 8)

    # Etape 2 : message reconstruit grâce à la réversibilité du XOR
    message = BitVecVal(0xA5, 8) ^ k

    # Etape 3 : message multiple de 4 (les 2 bits de poids faible sont nuls)
    solver = Solver()
    solver.add((message & BitVecVal(3, 8)) == BitVecVal(0, 8))

    # Etape 4 : résolution et extraction
    if solver.check() == sat:
        m = solver.model()
        clef = m[k].as_long()
        msg = m.eval(message).as_long()
        return {'clef': clef, 'message': msg}
    return None

clef = trouver_clef_xor()
print("Clef trouvee :", clef)

# Vérification
print(f"{clef['message']} ^ {clef['clef']} = {clef['message'] ^ clef['clef']:#x}")
print("Multiple de 4 :", clef['message'] % 4 == 0)

Clef trouvee : {'clef': 1, 'message': 164}
164 ^ 1 = 0xa5
Multiple de 4 : True


### Lecture du résultat

Le solveur rend `{'clef': 1, 'message': 164}` : la vérification confirme `164 ^ 1 = 0xa5` et `Multiple de 4 : True`. La contrainte `(message & 3) == 0` force les deux bits de poids faible de `k` — `0xA5` se terminant par `01`, la clef doit se terminer par `01` pour que le message se termine par `00`. La clef rendue n'est donc **pas unique** : Z3 s'arrête au premier modèle, et l'exercice suivant demande justement de les compter.

## Exercice 2 : Compter les clefs candidates

### Enonce

Un message de 8 bits a ete chiffre par XOR avec une clef `k`. Le message chiffre est `0xB7`. Cette fois, l'analyste sait deux choses du message clair : il est un **multiple de 4** (valeur entiere non signee) et sa valeur non signee est **strictement inferieure a 128**.

Écrivez `compter_clefs_possibles()` qui renvoie le **nombre** de clefs `k` (8 bits) compatibles avec ces deux indices. Verifiez votre reponse par une enumeration directe en Python (une boucle sur les 256 valeurs de `k`).

> **Indices :**
- La modelisation reste `message = BitVecVal(0xB7, 8) ^ k` ; les contraintes sont `(message & 3) == 0` et `ULT(message, BitVecVal(128, 8))`.
- Pour compter avec Z3 : resoudre, lire `m[k]`, ajouter `solver.add(k != m[k])` pour bloquer le modele trouve, et recommencer jusqu'a `unsat`.
- `0xB7` s'ecrit `10110111` : deux bits de `k` sont imposes par le multiple de 4, un par la borne a 128 — combien de bits restent libres ?

In [15]:
# EXERCICE 2 : Compter les clefs candidates.

def compter_clefs_possibles() -> int:
    """Compte les clefs k (8 bits) telles que message = 0xB7 ^ k soit multiple de 4 et < 128.

    Retourne : int (le nombre de clefs compatibles)

    # Indice : bloquez chaque modele trouve par solver.add(k != m[k]) jusqu'a unsat.
    # Etape 1 : declarer k = BitVec('k', 8) et message = BitVecVal(0xB7, 8) ^ k
    # Etape 2 : ajouter (message & 3) == 0 puis ULT(message, BitVecVal(128, 8))
    # Etape 3 : boucler check() -> modele -> bloquer, en comptant les modeles
    # Etape 4 : verifier par une enumeration directe sur les 256 valeurs de k
    """
    # TODO etudiant : implémentez le comptage
    return None  # TODO etudiant : remplacer par le nombre de clefs

nombre = compter_clefs_possibles()
print("Clefs possibles :", nombre)

# Tests attendus (decommentez apres implementation) :
# enumeration = sum(1 for k in range(256) if ((0xB7 ^ k) & 3) == 0 and (0xB7 ^ k) < 128)
# print("Enumeration directe :", enumeration)
# assert nombre == enumeration

Clefs possibles : None


## Exemple guidé 3 — Verifier une mise a jour de tableau (crédité #25642 — groupe 3 MIS : Alizée, Clémence et Pauline)

> Résolution originale : **groupe 3** (TP2, MIS, EPF), PR étudiante [#25642](https://github.com/jsboige/CoursIA/pull/25642). Code repris tel quel.

### Enonce

On dispose d'un tableau `scores` indexe par des entiers. On veut verifier la propriete suivante : si on effectue deux `Store` successifs sur des index différents `i` et `j` (avec `i != j`), l'ordre des Store n'a pas d'importance sur le résultat final.

Autrement dit, prouvez que pour tout tableau `a`, index `i != j` et valeurs `v1`, `v2` :
$$\text{Store}(\text{Store}(a, i, v_1), j, v_2) \text{ est équivalent a } \text{Store}(\text{Store}(a, j, v_2), i, v_1)$$

Verifiez cette propriete en tentant de trouver un contre-exemple (le solveur doit repondre `unsat`).

> **Indices :**
- Créez deux tableaux : `seq1 = Store(Store(a, i, v1), j, v2)` et `seq2 = Store(Store(a, j, v2), i, v1)`.
- Ajoutez la contrainte `i != j`.
- Ajoutez `Select(seq1, k) != Select(seq2, k)` pour un index `k` quelconque.
- Si `unsat`, la propriete est prouvee.

In [16]:
# EXEMPLE GUIDE 3 : Commutativite des Store sur des index differents (resolution groupe 3 MIS, TP2, #25642).

def verifier_commutativite_store() -> str:
    # Etape 1 : tableau d'entiers indexé par des entiers, et variables
    a = Array('a', IntSort(), IntSort())
    i, j, v1, v2, k = Ints('i j v1 v2 k')

    # Etapes 2 et 3 : les deux ordres de mise à jour
    seq1 = Store(Store(a, i, v1), j, v2)
    seq2 = Store(Store(a, j, v2), i, v1)

    # Etape 4 : on cherche un contre-exemple
    solver = Solver()
    solver.add(i != j)
    solver.add(Select(seq1, k) != Select(seq2, k))

    # Etape 5 : unsat = aucun contre-exemple = propriété prouvée
    if solver.check() == unsat:
        return 'prouve'
    return 'non_prouve'

resultat = verifier_commutativite_store()
print("Commutativite des Store :", resultat)

Commutativite des Store : prouve


### Lecture du résultat

Le solveur répond `prouve` : la requête de contre-exemple (`i != j` et `Select(seq1, k) != Select(seq2, k)`) est `unsat`, donc aucun modèle ne distingue les deux ordres d'écriture. En lecture, les deux séquences rendent bien la même valeur partout : en `k == i`, `seq1` rend `v1` (l'écriture en `j` ne touche pas `i`) et `seq2` rend aussi `v1` (dernière écriture en `i`) ; en `k == j`, les deux rendent `v2` ; partout ailleurs, `a[k]`. La commutativité tient **exactement parce que** `i != j` — l'exercice suivant fait tomber cette hypothèse.

## Exercice 3 : Faire tomber l'hypothese `i != j`

### Enonce

La preuve precedente repose entierement sur l'hypothese `i != j`. Supprimez-la : la propriete « l'ordre des deux `Store` n'a pas d'importance » est-elle encore vraie quand `i` et `j` peuvent etre egaux ?

Écrivez `tester_sans_inegalite()` qui cherche un contre-exemple **sans** la contrainte `i != j` (meme construction de `seq1` et `seq2`, meme requete `Select(seq1, k) != Select(seq2, k)`). Si le solveur repond `sat`, lisez le modele et renvoyez `{'statut': 'sat', 'i_egale_j': bool, 'v1': int, 'v2': int}` ; sinon renvoyez `{'statut': 'unsat'}`. Que remarque-t-on sur `i` et `j`, et sur `v1` et `v2`, dans le contre-exemple ?

> **Indices :**
- C'est le squelette de l'exemple guide 3, moins une contrainte.
- `m[i].as_long()` et `m[j].as_long()` lisent les valeurs du modele.
- Pour que les deux ordres different en `k == i == j`, la seconde ecriture doit ecraser la premiere avec une valeur differente.

In [17]:
# EXERCICE 3 : Faire tomber l'hypothese i != j.

def tester_sans_inegalite() -> dict:
    """Cherche un contre-exemple a la commutativite des Store SANS la contrainte i != j.

    Retourne : {'statut': 'sat', 'i_egale_j': bool, 'v1': int, 'v2': int} ou {'statut': 'unsat'}

    # Indice : reprenez la construction de l'exemple guide 3 et omettez la contrainte i != j.
    # Etape 1 : declarer a = Array('a', IntSort(), IntSort()) et i, j, v1, v2, k comme Int
    # Etape 2 : construire seq1 = Store(Store(a, i, v1), j, v2) et seq2 = Store(Store(a, j, v2), i, v1)
    # Etape 3 : ajouter uniquement Select(seq1, k) != Select(seq2, k)
    # Etape 4 : si sat, lire m[i], m[j], m[v1], m[v2] et remplir le dictionnaire
    """
    # TODO etudiant : implémentez la recherche de contre-exemple
    return None  # TODO etudiant : remplacer par le résultat

resultat = tester_sans_inegalite()
print("Sans l'inegalite i != j :", resultat)

# Tests attendus (decommentez apres implementation) :
# assert resultat['statut'] == 'sat'
# assert resultat['i_egale_j'] is True
# assert resultat['v1'] != resultat['v2']

Sans l'inegalite i != j : None


## Recapitulatif

Ce notebook a explore les mécanismes avances de Z3 au-dela du solveur de base :

| Concept | API Z3 | Usage |
|---------|--------|-------|
| Tactiques | `Tactic`, `Goal`, `Then`, `OrElse`, `Repeat` | Pretraiter, simplifier, decomposer les contraintes |
| Simplification | `simplify`, `ctx-solver-simplify` | Eliminer les redundances (syntaxique ou contextuelle) |
| Vecteurs de bits | `BitVec`, `UGT`, `ULT`, `Extract`, `Concat` | Arithmetic modulaire, opérations bit a bit, circuits |
| Tableaux | `Array`, `Select`, `Store` | Memoires fonctionnelles, mappings, verification de transitions |
| Solveurs specialises | `SolverFor('QF_LIA')`, `SolverFor('QF_BV')` | Exploiter la structure du problème pour accelerer la resolution |

**Points essentiels a retenir** :
1. Les **tactiques** offrent un contrôle fin sur la stratégie de resolution ; elles sont composees en pipelines.
2. Les `BitVec` modelisent le comportement exact du materiel (wrapping modulaire, opérations bit a bit, signe vs non signe).
3. Les `Array` sont des fonctions pures ; `Store` est immutable et `Select` lit la valeur.
4. `SolverFor` permet de sélectionner un solveur adapte au fragment logique du problème.

Les prochains notebooks exploreront les théories de chaînes (`String`, `Re`), les quantificateurs (`ForAll`, `Exists`) et l'optimisation avancee (objectifs multiples, Pareto).